# Subset selection with k-medoids + genetic algorithm (GA, intelligrate.subset)

This notebook shows a full subsetting workflow using the installed `intelligrate` package:
1) compute a distance matrix
2) inspect k diagnostics
3) fit k-medoids
4) run a genetic algorithm (GA) to select a balanced subset

It expects the example input files in `data/HF_sourdough/` relative to the folder where you start Jupyter, and writes outputs to `results/HF_sourdough/subset_100/`. You do not need to clone the full repository.


## Install and files
Install Intelligrate into the environment used by this notebook. Core Intelligrate is intended for Python 3.10-3.12 on macOS, Linux, and Windows.

If you already have Jupyter running from the environment you want to use:

```bash
pip install intelligrate
```

If you are creating a new notebook environment, install Jupyter in the same environment:

```bash
pip install intelligrate notebook ipykernel
```

Then download this notebook and the GitHub example folder `data/HF_sourdough/` into the same working folder. The expected layout is:

```text
your_working_folder/
  01_subset_kmedoids_ga_selection.ipynb
  data/HF_sourdough/feature_table_rel.tsv
  data/HF_sourdough/metadata.tsv
  results/                         # created by the notebook
```

The optional geographic basemap cells require:

```bash
pip install "intelligrate[maps]"
```


In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
pwd

## Environment check
This confirms the notebook kernel sees the pip-installed package.


In [ ]:
# Intelligrate should be installed in the active notebook kernel.
# From a terminal before launching Jupyter:
#   pip install intelligrate
# If this environment does not already have Jupyter:
#   pip install notebook ipykernel
# Optional map plots in the subset notebook:
#   pip install "intelligrate[maps]"

from importlib.metadata import version
import intelligrate
print('intelligrate version:', version('intelligrate'))


In [ ]:
# This notebook assumes it is run from a working folder with this structure:
# .
# |-- this_notebook.ipynb
# |-- data/HF_sourdough/...
# `-- results/                  # created automatically
#
# Download the notebook and the corresponding data/HF_sourdough/ folder from GitHub,
# then start Jupyter from this working folder. The package itself should come from pip.

project_dir = Path.cwd().resolve()
data_root = project_dir / 'data'
data_dir = data_root / 'HF_sourdough'
results_dir = project_dir / 'results' / 'HF_sourdough/subset_100'
results_dir.mkdir(parents=True, exist_ok=True)

required_files = ['feature_table_rel.tsv', 'metadata.tsv']
missing = [str(data_dir / name) for name in required_files if not (data_dir / name).exists()]
if missing:
    raise FileNotFoundError(
        'Missing example input files. Download the matching data folder from GitHub '
        'and keep it next to this notebook. Missing: ' + ', '.join(missing)
    )

print('Working folder:', project_dir)
print('Data folder:', data_dir)
print('Results folder:', results_dir)


## Load example inputs
- `feature_table_rel.tsv`: samples x features (relative abundances)
- `metadata.tsv`: sample metadata (includes latitude/longitude and categorical fields)


In [ ]:
feature_table = pd.read_csv(data_dir / 'feature_table_rel.tsv', sep='	', index_col=0)
metadata = pd.read_csv(data_dir / 'metadata.tsv', sep='	', index_col=0)

print('Feature table:', feature_table.shape)
print('Metadata:', metadata.shape)
feature_table.head()


## Step 1. Distance matrix
Parameters explained in simple words:
- `metric`: how we define distance between samples (`bray`, `jaccard`, `aitchison`)
- `assume_relative`: set True if the table is already relative abundance
- `pseudocount`: only used for Aitchison/CLR (small number to avoid log(0))

We use Bray-Curtis here, which is common for relative abundance tables.


DEAP is installed automatically with `intelligrate` because the genetic algorithm depends on it. If this import fails, install or reinstall Intelligrate in the active notebook environment before continuing.


In [ ]:
# DEAP is installed with intelligrate. This cell is intentionally non-executing.
# To repair a missing install in the current notebook kernel, run this in a terminal:
# pip install intelligrate


In [ ]:
from intelligrate.subset import compute_distance_matrix


metric = 'bray' #it is also possible to use 'jaccard' or 'aitchison'
assume_relative = True #only used for bray and jaccard, set to True if data is relative abundances
pseudocount = 1e-6 #only used for aitchison

D = compute_distance_matrix(
    feature_table,
    metric=metric,
    assume_relative=assume_relative,
    pseudocount=pseudocount,
)

# Save output
D.to_csv(results_dir / 'distance.tsv', sep='	')

print('Distance matrix shape:', D.shape)
D.iloc[:5, :5]


## Step 2. Choose k with diagnostics
Parameters explained:
- `k_range`: which k values to test
- `gap_B`: how many reference datasets to use for the gap statistic
- `random_state`: for reproducibility

We do not auto-pick k. Instead, we show diagnostics so you can choose.


In [ ]:
from intelligrate.subset import suggest_k


k_range = range(2, 30) #test k from 2 to 19
gap_B = 3 #number of bootstrap samples for gap statistic
random_state = 42

k_result = suggest_k(
    D,
    feature_table,
    k_range=k_range,
    gap_B=gap_B,
    random_state=random_state,
    return_fig=True,
)

kdiag = pd.DataFrame(
    {
        'k': k_result['k_values'],
        'silhouette': k_result['silhouette'],
        'davies_bouldin': k_result['davies_bouldin'],
        'gap': k_result['gap'],
        'gap_std': k_result['gap_std'],
    }
)

kdiag.to_csv(results_dir / 'k_diagnostics.tsv', sep='	', index=False)

# Plot the diagnostics (also saved below)
fig = k_result.get('figure')
#save as pdf:
fig.savefig(results_dir / 'k_diagnostics.pdf', bbox_inches='tight', dpi=300)
fig


In [ ]:
%matplotlib inline
fig

In [ ]:
k_result

## Step 3. Fit k-medoids
Parameters explained:
- `k`: number of clusters
- `random_state`: for reproducibility

K-medoids gives one representative "medoid" per cluster.


In [ ]:
from intelligrate.subset import fit_kmedoids

k = 10 #for now we take 10 clusters, but this should be chosen based on the dataset specific diagnostics above!!!!
random_state = 42

kmed = fit_kmedoids(D, k=k, random_state=random_state)
clusters = kmed['cluster_df']
cluster_counts = kmed['cluster_counts']
medoids = kmed['medoid_samples']

clusters.to_csv(results_dir / 'kmedoids_clusters.tsv', sep='	')
pd.DataFrame({'cluster': cluster_counts.index, 'n': cluster_counts.values}).to_csv(
    results_dir / 'kmedoids_cluster_counts.tsv', sep='	', index=False
)

print('Clusters:', clusters.shape)
clusters.head()


### Visualize clusters in 2D (MDS)
This is a quick 2D view of the distance matrix. It helps you see clustering structure.


In [ ]:
from sklearn.manifold import MDS

mds = MDS(n_components=2, dissimilarity='precomputed', random_state=42)
coords = mds.fit_transform(D.to_numpy(float))

plot_df = pd.DataFrame(coords, index=D.index, columns=['MDS1', 'MDS2'])
plot_df['Cluster'] = clusters['Cluster']
plot_df['Is_Medoid'] = clusters['Is_Medoid']

plt.figure(figsize=(7, 5))
for cl in sorted(plot_df['Cluster'].unique()):
    sub = plot_df[plot_df['Cluster'] == cl]
    plt.scatter(sub['MDS1'], sub['MDS2'], s=30, alpha=0.5, label=f'Cluster {cl}')

med = plot_df[plot_df['Is_Medoid']]
plt.scatter(med['MDS1'], med['MDS2'], s=80, facecolors='none', edgecolors='black', label='Medoids')

plt.title('MDS view of k-medoids clustering', fontsize=16)
plt.xlabel('MDS1', fontsize=14)
plt.ylabel('MDS2', fontsize=14)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=14)
plt.tight_layout()
#save as pdf:
plt.savefig(results_dir / 'kmedoids_mds.pdf', bbox_inches='tight', dpi=300)
plt.show()


## Step 4. GA subset selection
Parameters explained:
- `total_samples`: how many samples to select
- `balance_vars`: categorical metadata fields to balance, if you have numeric variables too, consider binning them first
- `coord_vars`: columns for latitude/longitude
- `population_size`, `generations`: GA runtime/quality tradeoff -> meaning how many sample sets per iteration and how many iterations
- `grid_weight`, `distance_weight`, `balance_weight`: how much each goal matters, higher = more important
- `min_category_n`: ignore metadata categories (within a variable) with fewer than this many samples
- `min_per_category`: enforce a minimum in the selected set, e.g. at least 2 samples per category per variable
- `fixed_include` / `fixed_exclude`: force include or exclude IDs

If you want to see every parameter in the function signature, check:
- the installed `intelligrate.subset.ga_subset` function


In [ ]:
metadata.columns.to_list()

In [ ]:
from intelligrate.subset import ga_subset

# Choose which metadata fields to balance
balance_vars = ['r_samp_country', 'r_samp_source', 'r_flour_type_red_red', 'r_flour_quality','cluster']
coord_vars = ('latitude', 'longitude')

# GA settings (smaller values run faster for this demo for generation and population sizes)
result_df, best_scores, fitness_array = ga_subset(
    cluster_df=clusters,
    metadata_df=metadata,
    total_samples=100,
    balance_vars=balance_vars,
    coord_vars=coord_vars,
    min_category_n=5,
    min_per_category=8,
    grid_size=2.5, #this is a large grid size to encourage more spread out samples, but this should be chosen based on the dataset specific diagnostics above!!!! (roughly 2.5x2.5 degree grid)
    population_size=2000,
    generations=100,
    random_state=42,
    fixed_include = ['vubh091', 'vubh077', 'vubh075', 'vubh076', 'ubzh106', 'ubzh035', 'ubzh037', 'ubzh051', 'ibbh058', 'ibbh028', '4c44e', '2cd6e', '5eadf', 'ubzh054b', 'ubzh006', 'ubzh033', 'vubh081'],
    fixed_exclude=[],
    metadata_weights={'r_samp_country': 1.0, 'r_samp_source': 1.0, 'r_flour_type_red_red': 1.0, 'r_flour_quality': 1.0, 'cluster': 1.0}, #weights for balancing different metadata fields, any values >=0
    grid_weight=3.0,
    distance_weight=2.0,
    balance_weight=1000.0,
    balance_scale=1.0,
    hard_penalty_weight=100.0,
    
)

result_df.to_csv(results_dir / 'ga_selected_samples.tsv', sep='	')
pd.DataFrame({'best_score': best_scores}).to_csv(results_dir / 'ga_best_scores.tsv', sep='	', index=False)
pd.DataFrame(fitness_array).T.to_csv(results_dir / 'ga_fitness_array.tsv', sep='	', index=False)

print('Selected samples:', result_df.shape)
result_df.head()


### GA diagnostics


In [ ]:
# Plot best fitness per generation
# Find the best generation and best score
best_gen_idx = np.argmax(best_scores)  # Index of the best generation (0-based)
best_gen = best_gen_idx + 1            # Generation number (1-based)
best_score_value = best_scores[best_gen_idx]
lowest_score_value = min(best_scores)
delta = best_score_value - lowest_score_value
all_fitnesses_array = np.array(fitness_array).T
generations = all_fitnesses_array.shape[1]
populations = all_fitnesses_array.shape[0]

plt.figure(figsize=(8, 4))
plt.plot(range(1, len(best_scores) + 1), best_scores, marker='o', color="#345084", label="Best score per generation")

# Add vertical line at best generation
plt.axvline(x=best_gen, color='#CB6BCEFF', linestyle='--', label=f"Max score: {best_score_value:.2f} (Gen. {best_gen}/{generations})")
# Add vertical line at best generation
plt.axvline(x=1, color='white', linestyle='--', alpha = 0.0, label=f"Score gain: +{delta:.2f}")
# Titles and labels
plt.title(f"Best score of n = {populations} individual subsets per generation", fontsize=16)
plt.xlabel("Generation", fontsize=14)
plt.ylabel("Score", fontsize=14)
#adjust fontsize of tick labels:
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.grid(False)
plt.legend(fontsize=14)
#despine:
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.tight_layout()
#save as pdf:
plt.savefig(results_dir / 'ga_best_fitness_per_generation.pdf', bbox_inches='tight', dpi=300)
plt.show()

In [ ]:
# Plot 2: Fitness volatility over generations
plt.figure(figsize=(8, 4))

all_fitnesses_array = np.array(fitness_array).T

for individual_fitness in all_fitnesses_array:
    plt.plot(range(1, generations + 1), individual_fitness, color='lightgray', alpha=0.7)
mean_fitness = np.mean(all_fitnesses_array, axis=0)
plt.plot(range(1, generations + 1), mean_fitness, color="#345084FF", linewidth=2, label="Mean score")
plt.title(f"Score volatility of n = {populations} individual subsets over generations", fontsize=16)
plt.xlabel("Generation", fontsize=14)
plt.ylabel("Score", fontsize=14)
plt.legend(fontsize=14)
plt.xticks(fontsize=14)
plt.yticks(fontsize=14)

plt.grid(False)
plt.gca().spines['top'].set_visible(False)
plt.gca().spines['right'].set_visible(False)
plt.tight_layout()

#save as pdf:
plt.savefig(results_dir / 'ga_fitness_volatility_per_generation.pdf', bbox_inches='tight', dpi=300)
plt.show()   


### Geographic view of the distribution



### Geographic view of the distribution

This optional map section requires additional geospatial plotting dependencies. Install them before starting the notebook if you want basemap plots:

```bash
pip install "intelligrate[maps]"
```

The core subset workflow above does not require these map dependencies. If they are unavailable, the map cells below skip cleanly and the rest of the notebook can still run.


In [ ]:
# Optional map dependencies. This cell is intentionally non-executing.
# Run this in a terminal before launching Jupyter if you want the geographic basemap section:
# pip install "intelligrate[maps]"


In [ ]:
#load some colors for plotting
hex_colors2 = ['#240E31FF', '#CB6BCEFF', '#468892FF', '#74F3D3FF',
              '#751C6DFF', '#FDC067FF', '#AC9ECEFF', '#6EC5ABFF']


In [ ]:
try:
    import geopandas as gpd
    from shapely.geometry import Point
    import matplotlib.pyplot as plt
    import matplotlib.colors as mcolors
    import contextily as ctx
    import numpy as np
    from matplotlib import cm
    HAS_MAP_DEPS = True
except ImportError as exc:
    HAS_MAP_DEPS = False
    print(
        "Skipping optional geographic basemap plot. Install map dependencies with: "
        "pip install \"intelligrate[maps]\""
    )

if HAS_MAP_DEPS:
    # === CONFIGURE THIS ===
    # DataFrame with selected samples, must include columns: Latitude, Longitude, Cluster
    df = result_df.copy()

    # Rename columns if needed
    df = df.rename(columns={'Latitude': 'latitude', 'Longitude': 'longitude'})

    # Variable to color by (e.g., 'Cluster' or 'sample_collected')
    group_var = 'Cluster'

    # === Convert to GeoDataFrame ===
    geometry = [Point(xy) for xy in zip(df['longitude'], df['latitude'])]
    geo_df = gpd.GeoDataFrame(df, geometry=geometry)
    geo_df = geo_df.set_crs(epsg=4326)  # WGS 84

    # Optional: Filter to Europe-ish bounding box
    geo_df = geo_df[(geo_df['longitude'] > -50) & (geo_df['longitude'] < 70)]
    geo_df = geo_df[(geo_df['latitude'] > 0) & (geo_df['latitude'] < 70)]

    # === Assign Colors Dynamically ===
    groups = sorted(geo_df[group_var].unique())
    cmap = cm.get_cmap('tab20', len(groups))
    hex_colors = [mcolors.rgb2hex(cmap(i)[:3]) for i in range(len(groups))]
    color_mapping = {grp: hex_colors[i] for i, grp in enumerate(groups)}

    # === Convert to Web Mercator for Basemap ===
    geo_df = geo_df.to_crs(epsg=3857)

    # === Plot ===
    fig, ax = plt.subplots(figsize=(8, 6))

    for grp in groups:
        sample_data = geo_df[geo_df[group_var] == grp]
        sample_data.plot(
            ax=ax,
            marker='o',
            color=color_mapping[grp],
            alpha=0.8,
            markersize=60,
            label=f"{group_var}: {grp}"
        )
    # Set fixed limits (Web Mercator units: meters, not degrees!)
    ax.set_xlim([-1700000, 4000000])  # roughly from -20°W to 40°E
    ax.set_ylim([4200000, 10500000])   # roughly from 30°N to 65°N
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronNoLabels)


    # Formatting
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc='upper left', fontsize=14, title='')
    plt.title("Geographic distribution of selected samples", fontsize=16)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.xlabel("")
    plt.ylabel("")
    plt.tight_layout()
    #save to pdf:
    plt.savefig(results_dir / 'selected_samples_map_by_cluster.pdf', bbox_inches='tight', dpi=300)
    plt.show()

    df_full = metadata.copy()   # <- all samples
    df_selected = result_df.copy()  # <- selected subset

    # Make sure columns match
    df_full = df_full.rename(columns={'Latitude': 'latitude', 'Longitude': 'longitude'})
    df_selected = df_selected.rename(columns={'Latitude': 'latitude', 'Longitude': 'longitude'})

    # Add 'Selected' flag
    df_full['Selected'] = df_full.index.isin(df_selected.index)

    # === Convert to GeoDataFrame ===
    geometry = [Point(xy) for xy in zip(df_full['longitude'], df_full['latitude'])]
    geo_df = gpd.GeoDataFrame(df_full, geometry=geometry)
    geo_df = geo_df.set_crs(epsg=4326)

    # Optional: Filter to Europe
    geo_df = geo_df[(geo_df['longitude'] > -50) & (geo_df['longitude'] < 70)]
    geo_df = geo_df[(geo_df['latitude'] > 0) & (geo_df['latitude'] < 70)]

    # Convert to Web Mercator
    geo_df = geo_df.to_crs(epsg=3857)

    # === Plot ===
    fig, ax = plt.subplots(figsize=(8, 6))

    # 1. Plot non-selected samples first (background)
    geo_df[geo_df['Selected'] == False].plot(
        ax=ax,
        marker='o',
        color=hex_colors2[5],
        markersize=30,
        alpha=0.8,
        label='Other sample'
    )
    # 2. Plot selected samples on top (foreground)
    geo_df[geo_df['Selected'] == True].plot(
        ax=ax,
        marker='o',
        color=hex_colors2[4],
        markersize=30,
        alpha=1.0,
        label='Selected sample'
    )

    # Set fixed zoom window (Europe)
    ax.set_xlim([-1700000, 4000000])
    ax.set_ylim([4200000, 10500000])

    # Add basemap
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.PositronNoLabels)

    # Formatting
    handles, labels = ax.get_legend_handles_labels()
    ax.legend(handles, labels, loc='upper left', fontsize=14, title='')
    plt.title("Geographic distribution of selected samples", fontsize=16)
    ax.set_xticks([])
    ax.set_yticks([])
    plt.xlabel("")
    plt.ylabel("")
    plt.tight_layout()
    #save to pdf:
    plt.savefig(results_dir / 'selected_samples_map_vs_all_samples.pdf', bbox_inches='tight', dpi=300)
    plt.show()


### Metadata balance plots
Compare selected vs full dataset for the key metadata fields.


In [ ]:
# Python
def plot_category_balance(full_df, selected_df, col, top_n=20):
    import matplotlib.pyplot as plt
    full_counts = full_df[col].value_counts().head(top_n)
    selected_counts = selected_df[col].value_counts().reindex(full_counts.index).fillna(0)
    plot_df = pd.DataFrame({
        'full': full_counts,
        'selected': selected_counts,
    })
    n_categories = len(full_counts.index)
    fig_width = 4 + n_categories * 0.25
    fig, ax1 = plt.subplots(figsize=(fig_width, 4.5))    
    # fig, ax1 = plt.subplots(figsize=(9, fig_width)) 
    plot_df['selected'].plot(kind='bar', ax=ax1, color=hex_colors2[4], position=0, width=0.4, label='Selected')
    ax1.set_ylabel('Selected count', fontsize=14)
    #also make x label larger:
    ax1.set_xlabel(col, fontsize=14)
    ax1.set_xticklabels(plot_df.index, rotation=45, ha='right', fontsize=14)
    ax1.tick_params(axis='y', labelsize=14)

    ax2 = ax1.twinx()
    plot_df['full'].plot(kind='bar', ax=ax2, color=hex_colors2[5], position=1, width=0.4, label='Full', alpha=0.5)
    ax2.set_ylabel('Full count', fontsize=14)
    ax2.tick_params(axis='y', labelsize=14)

    ax1.set_title(f'{col}: selected vs full (top {top_n})', fontsize=16)
    fig.tight_layout()
    plt.subplots_adjust(right=0.85)  # Add more space on the right
    #add legends
    #combine the legends from both axes:
    handles1, labels1 = ax1.get_legend_handles_labels()
    handles2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(handles1 + handles2, labels1 + labels2, loc='upper right', fontsize=14)
    n = len(plot_df.index)
    ax1.set_xlim(-0.5, n - 0.5 + 0.1)  # small extra padding on the right
    ax2.set_xlim(ax1.get_xlim())       # keep both axes aligned
    ax1.margins(x=0.03)
    ax2.margins(x=0.03)

    #save to pdf:
    plt.savefig(results_dir / f'category_balance_{col}.pdf', bbox_inches='tight', dpi=300)
    plt.show()

# Usage:
for col in balance_vars:
    plot_category_balance(metadata, result_df, col, top_n=20)


## Interpreting outputs
- `distance.tsv`: sample-sample distances used by all downstream steps
- `k_diagnostics.tsv` + plot: helps choose k
- `kmedoids_clusters.tsv`: cluster assignments and medoid flags
- `ga_selected_samples.tsv`: final subset with metadata and cluster label
- `ga_best_scores.tsv` + `ga_fitness_array.tsv`: GA convergence diagnostics
